In [ ]:
# Cell 1: Install Dependencies
!pip install --no-cache-dir --upgrade \
    scipy \
    ml_dtypes \
    unsloth==2026.1.4 \
    unsloth_zoo==2026.1.4 \
    transformers==4.57.6 \
    torch==2.10.0 \
    numpy==2.0.2 \
    pandas==2.2.2 \
    accelerate==1.12.0 \
    bitsandbytes==0.49.1 \
    trl==0.24.0

# Add this right after the installations to prevent the crash!
!pip uninstall -y torchcodec

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 277.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 238.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.7/405.7 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 kB 397.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 266.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 324.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 376.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 256.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 412.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

Found existing installation: torchcodec 0.11.0+cu128
Uninstalling torchcodec-0.11.0+cu128:
  Successfully uninstalled torchcodec-0.11.0+cu128


In [ ]:
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

In [ ]:
# Cell 2: Imports
import os
import re
import random
import numpy as np
import torch
from datasets import load_dataset
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from PIL import Image

# Cell 3: Set HF Token using Colab Secrets
import os
from google.colab import userdata



# Retrieve the secret from Google Colab
hf_token = userdata.get("hf-rw")
os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token)

# Cell 4: Load model and tokenizer via Unsloth
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Llama-3.2-11B-Vision-Instruct",
    load_in_4bit=False,
    torch_dtype=torch.bfloat16,
    use_gradient_checkpointing="unsloth",
    token=hf_token,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# Cell 5: Dataset Loading & Regex Setup
print("Loading V2 Dataset...")
dataset = load_dataset(
    "kazi420/ECG_LM_Final_Dataset_V3",
    token=hf_token,
)

LEADS = ['lead I', 'lead II', 'lead III', 'lead aVR', 'lead aVL', 'lead aVF',
         'lead V1', 'lead V2', 'lead V3', 'lead V4', 'lead V5', 'lead V6']

SEARCH_PATTERNS = {lead: re.compile(rf"{re.escape(lead)}:\s*([^,.\n]+)") for lead in LEADS}

# Cell 6: ECG-LM Optimized Augmentation Functions
def extract_leads_from_text(text_sequence):
    lead_descriptions = []
    for lead in LEADS:
        match = SEARCH_PATTERNS[lead].search(text_sequence)
        if match:
            desc = match.group(1).strip()
            lead_descriptions.append(f"{lead}: {desc}")
        else:
            lead_descriptions.append(f"{lead}: normal morphology")
    return ", ".join(lead_descriptions)

def apply_lead_masking_optimized(text_sequence, mask_prob=0.94):
    if random.random() > mask_prob:
        return extract_leads_from_text(text_sequence), extract_leads_from_text(text_sequence)

    k = random.randint(4, 6)
    leads_with_abnormality = []
    leads_normal = []
    lead_descriptions = {}

    for lead in LEADS:
        match = SEARCH_PATTERNS[lead].search(text_sequence)
        if match:
            description = match.group(1).strip()
            lead_descriptions[lead] = description
            if description.lower() == "normal morphology":
                leads_normal.append(lead)
            else:
                leads_with_abnormality.append(lead)
        else:
            lead_descriptions[lead] = "normal morphology"
            leads_normal.append(lead)

    max_abnormality_to_drop = max(0, len(leads_with_abnormality) - 1)
    abnormality_drop_count = min(k, max_abnormality_to_drop)
    normal_drop_count = min(k - abnormality_drop_count, len(leads_normal))

    leads_to_drop = []
    if abnormality_drop_count > 0:
        leads_to_drop.extend(random.sample(leads_with_abnormality, abnormality_drop_count))
    if normal_drop_count > 0:
        leads_to_drop.extend(random.sample(leads_normal, normal_drop_count))

    masked_leads = []
    full_leads = []

    for lead in LEADS:
        desc = lead_descriptions.get(lead, "normal morphology")
        full_leads.append(f"{lead}: {desc}")
        if lead not in leads_to_drop:
            masked_leads.append(f"{lead}: {desc}")
        else:
            masked_leads.append(f"{lead}: ")

    return ", ".join(masked_leads), ", ".join(full_leads)

def parse_ecg_text(text_sequence):
    text_sequence = text_sequence.replace("<ECG_IMAGE_TOKEN>", "").strip()
    if "Findings:" in text_sequence:
        patient_profile, rest = text_sequence.split("Findings:", 1)
        patient_profile = patient_profile.strip()
        findings_match = re.search(r"Findings:\s*([^.]+\.)", text_sequence)
        diagnosis_match = re.search(r"Diagnosis:\s*([^.]+\.)", text_sequence)
        findings_text = findings_match.group(1).strip() if findings_match else ""
        diagnosis_text = diagnosis_match.group(1).strip() if diagnosis_match else ""
        ecg_report = f"{findings_text} {diagnosis_text}".strip()
    else:
        patient_profile = text_sequence.split(".")[0]
        ecg_report = text_sequence

    rhythm_match = re.search(r"rhythm \((\d+)\s*bpm\)", text_sequence)
    rhythm_info = rhythm_match.group(0) if rhythm_match else ""
    return patient_profile, ecg_report, rhythm_info

def convert_to_conversation(sample):
    original_seq = sample["text_sequence"]

    # 1. Parse into ECG-LM format
    patient_profile, ecg_report, rhythm_info = parse_ecg_text(original_seq)

    # 2. Apply lead masking
    masked_leads, full_leads = apply_lead_masking_optimized(original_seq)

    # 3. Construct Prompts
    user_prompt = (
        f"{patient_profile}. "
        f"Analyze the ECG image and generate a complete clinical report. "
        f"Some leads are masked. Use visual evidence to complete them.\n\n"
        f"Visible Leads: {masked_leads}"
    )

    assistant_target = f"ECG Report: {ecg_report}. Lead descriptions: {full_leads}"

    # Force strictly to PIL Image to prevent dictionary/type mismatch errors
    if isinstance(sample["ecg_image"], dict):
        img_data = sample["ecg_image"].get("bytes") or sample["ecg_image"].get("path")
        # Handle accordingly if it's raw bytes, usually huggingface resolves this automatically
        pass

    img = sample["ecg_image"].convert("RGB")

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_prompt},
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": assistant_target}
            ]
        },
    ]
    return {"messages": conversation}

# Apply mapping
print("Converting datasets to Unsloth conversational format...")
converted_train = [convert_to_conversation(s) for s in dataset["train"]]
converted_val   = [convert_to_conversation(s) for s in dataset["validation"]]

# Cell 7: Training setup
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=converted_train,
    eval_dataset=converted_val,
    args=SFTConfig(
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        warmup_steps=15,

        # ⚠️ max_steps=564 মুছে ফেলা হয়েছে
        num_train_epochs=5.0,       # 🎯 ২ ইপক যোগ করা হলো

        learning_rate=2e-4,
        fp16=False,  # 🚫 FP16 বন্ধ করুন
        bf16=True,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="/content/llama-ecg-final",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        dataset_num_proc=2,
        max_seq_length=2048,

        # 🎯 ভ্যালিডেশন এবং সেভ করার স্ট্র্যাটেজি "epoch" করা হলো
        eval_strategy="epoch",      # প্রতি ইপক শেষে ভ্যালিডেশন করবে
        save_strategy="epoch",      # প্রতি ইপক শেষে মডেল সেভ করবে

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    ),
)

# Cell 8: Train
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")

trainer_stats = trainer.train()

# Cell 9: Push to HuggingFace
repo_id = "bappy2001/Llama-3.2-11B-ECG-LM-Final_V3"

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print("Successfully pushed to HuggingFace:", repo_id)

==((====))==  Unsloth 2026.1.4: Fast Mllama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Unsloth: Making `model.base_model.model.model.vision_model.transformer` require gradients
Loading V2 Dataset...


Generating train split:   0%|          | 0/5532 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,532 | Num Epochs = 5 | Total steps = 1,730
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 67,174,400 of 10,737,395,235 (0.63% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.145500,0.135695
2,0.136700,0.130283
3,0.120100,0.128323
4,0.115100,0.128410
5,0.110000,0.131532


Unsloth: Not an error, but MllamaForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Saved model to https://huggingface.co/bappy2001/Llama-3.2-11B-ECG-LM-Final_V3
Successfully pushed to HuggingFace: bappy2001/Llama-3.2-11B-ECG-LM-Final_V3
